In [ ]:
import os
from tqdm import tqdm
import json
import pandas as pd


exp = []
for filename in tqdm(os.listdir("INSERT YOURS/explanations/")):
    with open(os.path.join("INSERT YOURS/explanations/", filename)) as f:
        lines = f.readlines()
        for line in lines:
            exp.append(json.loads(line))

exp_df = []
for feat in exp:
    exp_df.append({
        "index": int(feat["index"]),
        "desc": feat["description"],
    })

exp_df = pd.DataFrame(exp_df)
exp_df = exp_df[~ exp_df["index"].duplicated()]

exp_df

In [ ]:
def create_request_line(feature_idx, feature_desc, tag):
    messages = [
        {
            "role": "user",
            "content": (
                f"Here is the description of a certain textual property: \"{feature_desc.strip()}\". "
                f"Is this property related to {tag}? "
                "Respond with one word only: yes or no."
            )
        },
    ]

    return {
        "custom_id": f"{feature_idx}-{tag.replace(' ', '_')}", 
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4.1-nano-2025-04-14", 
            "messages": messages, 
            "max_completion_tokens": 3
        }
    }

tags = [
    "computer code, programming languages, or math",
    "syntax or text structure"
]

lines = []
for tag in tags:
    for _, row in exp_df.iterrows():
        lines.append(create_request_line(row["index"], row["desc"], tag))

In [ ]:
import json

with open("INSERT YOURS.jsonl", "w") as f:
    for line in lines:
        f.write(json.dumps(line) + "\n")


In [ ]:
OPENAI_KEY = "INSERT YOURS"

In [ ]:

from openai import OpenAI
client = OpenAI(api_key=OPENAI_KEY)

batch_input_file = client.files.create(
    file=open("INSERT YOURS.jsonl", "rb"),
    purpose="batch"
)

batch_input_file_id = batch_input_file.id
batch = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "feature tagging"
    }
)

batch

In [ ]:
batch = client.batches.retrieve("INSERT YOURS")
print(batch)
print(batch.status)

In [ ]:
file_response = client.files.content("INSERT YOURS")

In [ ]:
def parse_response_line(line):
    unjsoned = json.loads(line)
    custom_id = unjsoned["custom_id"]
    feature_idx = int(custom_id.split("-")[0])
    tag = custom_id.split("-")[1].replace("_", " ")
    response = unjsoned["response"]["body"]["choices"][0]["message"]["content"]
    return {
        "feature_idx": feature_idx,
        "tag": tag,
        "response": response.strip().lower()
    }

In [ ]:
results = [parse_response_line(line) for line in file_response.iter_lines()]
results = pd.DataFrame(results)

In [ ]:
results["response"].value_counts()

In [ ]:
results_is_code = results[results["tag"] == "computer code, programming languages, or math"][["feature_idx", "response"]]
results_is_structure = results[results["tag"] == "syntax or text structure"][["feature_idx", "response"]]

In [ ]:
exp_df_w_code = exp_df.join(results_is_code.set_index("feature_idx"), on="index")
exp_df_w_code = exp_df_w_code.rename(columns={"response": "is_code"}).join(results_is_structure.set_index("feature_idx"), on="index")
exp_df_final = exp_df_w_code.rename(columns={"response": "is_syntax"})
exp_df_final

In [ ]:
exp_df_final["is_code"] = (exp_df_final["is_code"] == "yes").astype(int)
exp_df_final["is_syntax"] = (exp_df_final["is_syntax"] == "yes").astype(int)


In [ ]:
exp_df_final.to_csv("INSERT YOURS.csv", index=False)